# Fortran + LaTeX 笔记模板

本 Notebook 展示如何将 Fortran 代码学习、数值实验和 LaTeX 笔记无缝结合。

## 工作流
1. **学习**：阅读 Fortran 源码，理解算法
2. **复现**：在 Jupyter 中用小代码验证
3. **可视化**：用 Python 绘图，保存为 PDF
4. **笔记**：导出到 LaTeX，或复制关键部分

## 与 LaTeX 的结合方式
- **方式 1**：用 `nbconvert` 转成 `.tex`，然后 `\input{}`
- **方式 2**：截图/保存图表，在 LaTeX 中 `\includegraphics`
- **方式 3**：直接在这个 Notebook 中写 Markdown + LaTeX 公式

In [ ]:
# 初始化环境
%load_ext fortranmagic
import numpy as np
import matplotlib.pyplot as plt

# 设置 matplotlib 输出为 PDF（方便 LaTeX 使用）
plt.rcParams['savefig.format'] = 'pdf'
plt.rcParams['font.size'] = 12

print("环境初始化完成！")

## 示例：复现 `CDFT_density.f90` 的核心算法

### 理论基础

在 CDFT 中，密度算符定义为：

$$
\rho(\mathbf{r}) = \sum_i n_i \psi_i^*(\mathbf{r}) \psi_i(\mathbf{r})
$$

其中 $n_i$ 是占据数，$\psi_i$ 是单粒子波函数。

### 数值实现

在 `CDFT_density.f90` 中，密度计算在柱坐标下进行：
- 使用高斯积分（`ngh × ngl` 网格）
- 对 K-block 进行并行计算
- 输出到 `./output/` 目录

In [ ]:
%%fortran
! 简化版密度计算（从 CDFT_density.f90 提取）
module SimpleDensity
    implicit none
    integer, parameter :: dp = kind(1.0d0)
contains
    
    ! 计算对角密度（简化版）
    subroutine compute_diag_density(rho, psi, n_states, n_grid)
        integer, intent(in) :: n_states, n_grid
        real(dp), intent(in)  :: psi(n_grid, n_states)  ! 波函数
        real(dp), intent(out) :: rho(n_grid)            ! 密度
        integer :: i, j
        
        rho = 0.0_dp
        do j = 1, n_states
            do i = 1, n_grid
                rho(i) = rho(i) + psi(i, j)**2  ! 假设占据数 = 1
            end do
        end do
    end subroutine
    
    ! 计算总粒子数（验证归一化）
    real(dp) function total_particles(rho, dr, n_grid)
        integer, intent(in) :: n_grid
        real(dp), intent(in) :: rho(n_grid), dr
        integer :: i
        
        total_particles = 0.0_dp
        do i = 1, n_grid
            total_particles = total_particles + rho(i) * dr
        end do
    end function
end module SimpleDensity

In [ ]:
# 测试简化版密度计算
from fortran import SimpleDensity
import numpy as np

# 参数设置
n_grid = 200
n_states = 10
r_max = 10.0  # fm
dr = r_max / n_grid

# 生成测试波函数（谐振子基）
r = np.linspace(0, r_max, n_grid)
psi = np.zeros((n_grid, n_states))

for j in range(n_states):
    n = j + 1
    psi[:, j] = r * np.exp(-0.5 * (r / 2.0)**2)  # 简化波函数
    psi[:, j] = psi[:, j] / np.sqrt(np.sum(psi[:, j]**2) * dr)  # 归一化

# 调用 Fortran 计算密度
rho = np.zeros(n_grid)
SimpleDensity.compute_diag_density(rho, psi, n_states, n_grid)

# 验证粒子数
N = SimpleDensity.total_particles(rho, dr, n_grid)
print(f"总粒子数: {N:.4f} (应为 {n_states})")
print(f"归一化误差: {abs(N - n_states):.2e}")

In [ ]:
# 可视化密度分布
plt.figure(figsize=(10, 6))
plt.plot(r, rho, linewidth=2, label='密度 $\rho(r)$')
plt.axhline(y=0, color='k', linestyle=':', alpha=0.5)
plt.xlabel('$r$ (fm)')
plt.ylabel('$\rho(r)$ (fm$^{-3}$)')
plt.title('简化版密度分布（Fortran 计算）')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()

# 保存为 PDF（用于 LaTeX）
plt.savefig('fig_density_simple.pdf', dpi=300)
print("图表已保存: fig_density_simple.pdf")

## 导出到 LaTeX

### 方法 1：直接转换 Notebook

```bash
jupyter nbconvert --to latex Fortran_Learning.ipynb
```

生成的 `.tex` 文件可以直接 `\input` 到你的主文档：

```latex
\input{Fortran_Learning.tex}
```

### 方法 2：手动整理关键部分

复制以下部分到你的 LaTeX 笔记：

1. **理论公式**（从上面的 Markdown 单元格）
2. **Fortran 代码**（可以用 `listings` 宏包高亮）
3. **图表**（用 `\includegraphics` 引用保存的 PDF）

### 示例 LaTeX 代码

```latex
\section{密度计算的数值实现}
\label{sec:density_numerics}

算法\ref{alg:density}展示了密度计算的简化版本。

\begin{equation}
    \rho_i = \sum_{j=1}^{N_{states}} |\psi_j(r_i)|^2
\end{equation}

\begin{figure}[h]
    \centering
    \includegraphics[width=0.8\\textwidth]{fig_density_simple.pdf}
    \caption{密度分布的数值结果}
    \label{fig:density}
\end{figure}
```

In [ ]:
# 高级：用 F2PY 封装成真正的 Python 模块
import subprocess
import os

# 创建 Fortran 源文件
fortran_code = '''
subroutine advanced_density(rho, r, A, n)
    implicit none
    integer, intent(in) :: n, A
    real(8), intent(in)  :: r(n)
    real(8), intent(out) :: rho(n)
    real(8) :: R, rho0, pi
    integer :: i
    
    pi = 3.141592653589793d0
    R = 1.2d0 * A**(1.0d0/3.0d0)
    rho0 = 3.0d0 * A / (4.0d0 * pi * R**3)
    
    do i = 1, n
        if (r(i) <= R) then
            rho(i) = rho0
        else
            rho(i) = 0.0d0
        end if
    end do
end subroutine
'''

# 写入文件
with open('density_module.f90', 'w') as f:
    f.write(fortran_code)

print("Fortran 模块已保存: density_module.f90")
print("可以用以下命令编译:")
print("  python -m numpy.f2py -c density_module.f90 -m density_mod")

## 总结与下一步

### 已完成
- ✅ 在 Jupyter 中运行 Fortran 代码
- ✅ 验证算法正确性（粒子数守恒）
- ✅ 生成出版级图表（PDF）

### 下一步
1. **深入阅读**：`CDFT_main.f90` → 理解主流程
2. **复现更多模块**：
   - `CDFT_expectation.f90` → 期望值计算
   - `CDFT_force.f90` → 自洽场迭代
3. **封装成包**：用 F2PY 把常用子程序编译成 `.pyd`
4. **整合到 LaTeX**：选择方式 1 或 2，更新你的笔记